# Imports

In [ ]:
import sys
sys.path.append("") # local utils.py path
import utils
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import pairwise_distances

## Confused by the parameter names? Try *help(utils.function_name)*

**Example**

In [ ]:
help(utils.plot_keypoint_tracking)

# Define Tm Barcode_Target Columns

In [ ]:
tmB_T = ["LowTm", "HighTm"]

## Load Predicted Tm Barcode_Target data

In [ ]:
pred_c1 = pd.read_csv("../../pred_Tmb/channel_name.csv")

# Load Saved Files from Image Processing Pipeline

### Note: Probe Combos are combinations of Tm Probe values and Probe color channels. For example, (Hex_TmP1, Cy5_TmP2) is a probe combo where Hex is the color channel for Tm Probe 1 and Cy5 is the color channel for Tm Probe 2.

## Area 1

In [ ]:
par_dir = ""

In [ ]:
a1_c1 = pd.read_csv(par_dir+"chip1-1_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 2

In [ ]:
a2_c1 = pd.read_csv(par_dir+"chip1-2_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 3

In [ ]:
a3_c1 = pd.read_csv(par_dir+"chip1-3_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 4

In [ ]:
a4_c1 = pd.read_csv(par_dir+"chip2_1_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 5

In [ ]:
a5_c1 = pd.read_csv(par_dir+"chip2_2_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 6

In [ ]:
a6_c1 = pd.read_csv(par_dir+"chip3-1_B1R1_bkg_subtracted.csv") # Probe Combo 1

## Area 7

In [ ]:
a7_c1 = pd.read_csv(par_dir+"chip3-2_B1R1_bkg_subtracted.csv") # Probe Combo 1

# Probe Combo 1 (Hex_TmP1, Cy5_TmP2)

In [ ]:
aligned_c1, c1_shift = utils.align_datasets(data1 = a1_c1, data2 = [a2_c1, a3_c1,
                                                                         a4_c1, a5_c1, a6_c1, a7_c1], 
                                     size=120, max_shift_x=2, max_shift_y=2, 
                                    cols_to_align=['LowTm', 'HighTm'], manual=True)

## Imaging Area Data Integration and Batch Calibration

In [ ]:
importlib.reload(utils)

In [ ]:
aligned_a2_c1 = utils.apply_global_shift(aligned_c1[0], shift=c1_shift[0], cols_to_shift=["LowTm", "HighTm"])
aligned_a3_c1 = utils.apply_global_shift(aligned_c1[1], shift=c1_shift[1], cols_to_shift=["LowTm", "HighTm"])
aligned_a4_c1 = utils.apply_global_shift(aligned_c1[2], shift=c1_shift[2], cols_to_shift=["LowTm", "HighTm"])
aligned_a5_c1 = utils.apply_global_shift(aligned_c1[3], shift=c1_shift[3], cols_to_shift=["LowTm", "HighTm"])
aligned_a6_c1 = utils.apply_global_shift(aligned_c1[4], shift=c1_shift[4], cols_to_shift=["LowTm", "HighTm"])
aligned_a7_c1 = utils.apply_global_shift(aligned_c1[5], shift=c1_shift[5], cols_to_shift=["LowTm", "HighTm"])

In [ ]:
c1_nline_x = 3
c1_nline_y = 8
pred_c1_grid = utils.create_grid(pred_c1[tmB_T], c1_nline_x, c1_nline_y)

In [ ]:
all_c1 = np.vstack([a1_c1[tmB_T],
                    aligned_a2_c1[tmB_T], 
                    aligned_a3_c1[tmB_T],
                    aligned_a4_c1[tmB_T], 
                    aligned_a5_c1[tmB_T],
                    aligned_a6_c1[tmB_T],
                   aligned_a7_c1[tmB_T],
                   ])

In [ ]:
all_c1_grid = utils.create_grid(all_c1, c1_nline_x, c1_nline_y,manual= False)

In [ ]:
pred_c1_transformed, pred_shift = utils.align_datasets(data1 = a1_c1, data2 = pred_c1,
                                     size=120, max_shift_x=1, max_shift_y=1, 
                                    cols_to_align=['LowTm', 'HighTm'], manual=True) #Choose Manual=True if needed

In [ ]:
pred_c1_transformed = utils.apply_global_shift(pred_c1, shift=pred_shift[0], cols_to_shift=["LowTm", "HighTm"])

In [ ]:
_, pred_c1_shift = utils.align_datasets(data1 = a1_c1, data2 = pred_c1_transformed,
                                     size=120, max_shift_x=2, max_shift_y=2, 
                                    cols_to_align=['LowTm', 'HighTm'], manual=True)

In [ ]:
pred_c1_transformed = utils.apply_global_shift(pred_c1_transformed, shift=pred_c1_shift[0], cols_to_shift=["LowTm", "HighTm"])

In [ ]:
utils.scatter_plot_dfs([pred_c1_transformed, a1_c1, 
                        aligned_a2_c1,aligned_a3_c1,
                        aligned_a4_c1,aligned_a5_c1,
                        aligned_a6_c1, aligned_a7_c1
                       ], "LowTm", "HighTm")

## Greedy Surjective Matching with Prediction

In [ ]:
importlib.reload(utils)

In [ ]:
concatenated_c1 = utils.process_df_list([a1_c1, aligned_a2_c1, aligned_a3_c1, 
                                       aligned_a4_c1, aligned_a5_c1,
                        aligned_a6_c1, aligned_a7_c1
                                        ])

In [ ]:
c1_dist_mat = pairwise_distances(concatenated_c1[["LowTm", "HighTm"]], pred_c1_transformed[["LowTm", "HighTm"]])
concatenated_c1_idx_list = concatenated_c1.index.tolist()
c1_gene_names = pred_c1_transformed["gene_name"].tolist()

In [ ]:
c1_dist_matL, c1_dist_matH = utils.compute_distance_matrix(concatenated_c1[["LowTm", "HighTm"]], pred_c1_transformed[["LowTm", "HighTm"]])

In [ ]:
pred_c1_transformed = pred_c1_transformed[pred_c1_transformed.gene_name != "B1R1-NCAM1"]

In [ ]:
c1_dist_mat = pairwise_distances(concatenated_c1[["LowTm", "HighTm"]], pred_c1_transformed[["LowTm", "HighTm"]])
concatenated_c1_idx_list = concatenated_c1.index.tolist()
c1_gene_names = pred_c1_transformed["gene_name"].tolist()

In [ ]:
utils.scatter_plot_dfs([pred_c1_transformed, a1_c1, 
                        aligned_a2_c1,aligned_a3_c1,
                        aligned_a4_c1
                       ], "LowTm", "HighTm")

In [ ]:
c1_matching = utils.greedy_surjective_constrained_matching(concatenated_c1_idx_list, c1_gene_names, c1_dist_mat, 1.2)

In [ ]:
utils.plot_matching(c1_matching, concatenated_c1, ["LowTm","HighTm"])

## Clustering and Quantification

In [ ]:
importlib.reload(utils)

In [ ]:
new_matching_df_c1, _ = utils.refine_clusters(matching_df=c1_matching, external_df= concatenated_c1, 
                                  selected_column_names = ["LowTm","HighTm"], model="GMM")

In [ ]:
utils.plot_matching(new_matching_df_c1, concatenated_c1, ["LowTm","HighTm"])

In [ ]:
concatenated_c1

In [ ]:
pc_result1 = utils.plot_matching_interactive(matching_df=new_matching_df_c1, external_df=concatenated_c1, 
                                   selected_column_names=["LowTm", "HighTm"],
                                   std_x=0.5, std_y=0.25, off_diag=0)

In [ ]:
pc1_stats = pc_result1["stats_df"]

## Note: Add additional Probe Combos as needed (Repeats of above sections)

# Visualization

In [ ]:
# Add other color/name options here
color_list = ["red", 
            #   "gold", "orange", "chocolate"
              ]
name_list = ["68.37 R ; 56.55 Y",
    # "68.37 R ; Na   ", "Na   ; 77.78 Y",   "44.36 R ; 77.78 Y"
    ]

In [ ]:
df_list = [pc1_stats]

In [ ]:
utils.plot_tm_levels(df_list, name_list, color_list, plot_size=(15,100), size_scale=5, z_spacing=30)

# Save Analysis Outputs

In [ ]:
updated_matching_df = pd.concat([pc_result1["updated_matching_df"]], axis=1)
cluster_dict = {}
for col in updated_matching_df.columns:
    for idx_tuple in updated_matching_df[col]:
        if isinstance(idx_tuple, tuple) and len(idx_tuple) == 2:
            idx = int(idx_tuple[1])
            cluster_dict[idx] = col

In [ ]:
updated_all_c1s = utils.update_cluster_assignment(pc_result1["updated_matching_df"], [a1_c1, aligned_a2_c1,
                                                                                     aligned_a3_c1, aligned_a4_c1,
                                                                                     aligned_a5_c1, aligned_a6_c1, aligned_a7_c1
                                                                                     ])
aligned_all_c1 = pd.concat(updated_all_c1s, axis=0)

In [ ]:
aligned_all_c1 = aligned_all_c1.reset_index(drop=True)
aligned_all_c1['idx'] = aligned_all_c1.index

aligned_all_c1.to_csv("aligned_all_c1.csv", index=True)

In [ ]:
pc1_stats.to_csv("B1R1_stats.csv")

In [ ]:
combined_stats = pd.concat([pc1_stats])
new_names = combined_stats["cluster_name"].to_list()
new_names =[name[5:] for name in new_names]
combined_stats["cluster_name"] =new_names
combined_stats.to_csv("combined_stats.csv")